# 🚀 Huấn Luyện & Đánh Giá Các Mô Hình Cơ Sở pyKT Trên Tập Dữ Liệu Eedi (2x NVIDIA Tesla T4)

**Các mô hình cơ sở:** DKT (Deep Knowledge Tracing), DKVMN (Dynamic Key-Value Memory Networks), SAKT (Self-Attentive Knowledge Tracing)  
**Thư viện nền tảng:** `pyKT` (`pykt-toolkit`) — Khởi tạo nguyên bản 100% từ `pykt.models` với không gian động **`num_c = 57` kỹ năng lá cấp 3 (`0..56`)**  
**Tập dữ liệu:** **Eedi NeurIPS 2020 (Task 3 & 4)** — `948` câu hỏi (`934` đơn kỹ năng, `14` đa kỹ năng), `4.918` học sinh huấn luyện (5-Fold CV đồng bộ `ACD-MMKTv3`), `1.230` học sinh kiểm thử độc lập (`124.960` lượt `unified` = `74.276` `public` + `50.684` `private`).  
**Tối ưu hóa phần cứng:** Hỗ trợ chạy song song cụm **2x NVIDIA Tesla T4 GPUs** (`--dual-gpu`: Worker 0 trên `cuda:0` chạy Folds `[0, 1, 2]`, Worker 1 trên `cuda:1` chạy Folds `[3, 4]`), cô lập toàn bộ kết quả vào phân vùng `outputs/*/eedi/`.

---

### ⚙️ Hướng Dẫn Cấu Hình Kaggle Notebook (Bắt Buộc):
1. **Accelerator:** Trong menu **Session options** bên phải, chọn **GPU T4 x 2** (hoặc GPU T4 x 1 / P100 nếu chỉ có 1 GPU).
2. **Internet:** Bật **Internet ON** để cài đặt `pykt-toolkit` và clone/pull mã nguồn từ GitHub.
3. **Data Input:** Nhấn **+ Add Input**, gắn Dataset **Eedi đã tiền xử lý** (chứa thư mục `pykt_kc_level/` hoặc dữ liệu `data/processed/Eedi/` + `splits/` + dữ liệu thô Eedi để notebook tự sinh `pykt_kc_level/` nếu chưa có).
   *(Tùy chọn: Có thể gắn thêm Dataset chứa checkpoint/report `ACD-MMKTv3` trên Eedi để đối sánh trực tiếp và tái tổng hợp bảng kiểm định thống kê bắt cặp ngay trong notebook).*
4. **Tùy biến siêu tham số:** Chỉnh sửa trực tiếp `TARGET_MODEL`, `NUM_EPOCHS`, `BATCH_SIZE`, `FOLDS_TO_RUN` tại **Cell [6]**.

In [ ]:
# [1] Kiểm tra cấu hình phần cứng & Cấu hình môi trường Dual GPU T4
import os
import sys
import torch

# Giảm phân mảnh bộ nhớ CUDA và cố định thứ tự PCI Bus ID
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

print(f"🔥 PyTorch Version: {torch.__version__}")
is_gpu = torch.cuda.is_available()
gpu_count = torch.cuda.device_count() if is_gpu else 0

if is_gpu:
    print(f"⚡ [ACCELERATOR: NVIDIA GPU] Đã nhận diện cụm {gpu_count} GPUs:")
    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)
        print(f"   - [GPU {i}] {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB | Compute Capability: {props.major}.{props.minor}")
    print("\n--- Thông tin chi tiết nvidia-smi ---")
    !nvidia-smi
else:
    print("⚠️ Không tìm thấy GPU CUDA! Đang chạy trên CPU (tốc độ sẽ chậm hơn đáng kể).")
    print("👉 Vui lòng chọn Accelerator là 'GPU T4 x 2' trong menu Session Options bên phải!")

HAS_DUAL_GPU = (gpu_count >= 2)
print(f"\nChế độ Dual-GPU khả dụng: {HAS_DUAL_GPU}")

In [ ]:
# [2] Thiết lập môi trường làm việc & Clone/Pull repository từ GitHub
import os
import sys

REPO_URL = "https://github.com/nbao0208/kt-mmkt-research.git"
REPO_DIR = "/kaggle/working/kt-mmkt-research"

if os.path.exists("src/models/baselines/pykt_models.py") and os.path.exists("src/data/eedi_pykt_preprocessor.py"):
    print("✅ Bạn đang đứng trực tiếp trong thư mục repository kt-mmkt-research.")
    !git pull origin main
elif os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"✅ Đã chuyển thư mục làm việc đến: {REPO_DIR}")
    !git pull origin main
else:
    print(f"📥 Đang clone repository từ {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    print(f"✅ Đã clone thành công và chuyển vào: {os.getcwd()}")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Working Directory:", os.getcwd())

In [ ]:
# [3] Cài đặt & cập nhật các thư viện phụ thuộc (bao gồm pykt-toolkit)
!pip install -q -r requirements.txt
!pip install -q "pykt-toolkit>=0.1.7"
!pip install -q seaborn matplotlib scikit-learn tabulate pandas scipy pyyaml

import pykt
print(f"✅ pyKT Toolkit Version: {getattr(pykt, '__version__', 'Installed')}")
print("✅ Hoàn tất cài đặt toàn bộ gói phụ thuộc cho pyKT trên Eedi!")

In [ ]:
# [4] Tự động phát hiện, liên kết (Symlink) và chuẩn bị dữ liệu Eedi (Task 3 & 4) cho pyKT
import glob
import os
import subprocess
import sys
from pathlib import Path

def safe_symlink(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    dst.symlink_to(src)

# 1. Liên kết dữ liệu Eedi đã tiền xử lý (.pt, .json) vào data/processed/Eedi/
proc_dir = Path("data/processed/Eedi")
proc_dir.mkdir(parents=True, exist_ok=True)
candidates_proc = glob.glob("/kaggle/input/**/sequences_test_public.pt", recursive=True)
if not candidates_proc:
    candidates_proc = [p for p in glob.glob("/kaggle/input/**/static_item_bank.pt", recursive=True) if "eedi" in p.lower()]

linked_proc = 0
if candidates_proc:
    src_proc_dir = Path(candidates_proc[0]).parent
    print(f"🔍 Tìm thấy dữ liệu processed Eedi tại: {src_proc_dir}")
    for ext in ["*.pt", "*.json", "*.npy", "*.parquet"]:
        for src_file in src_proc_dir.glob(ext):
            safe_symlink(src_file, proc_dir / src_file.name)
            linked_proc += 1
print(f"✅ Đã liên kết {linked_proc} tệp vào {proc_dir}")

# 2. Liên kết các tệp phân chia 5-Fold & Test UIDs vào data/splits/Eedi/
splits_dir = Path("data/splits/Eedi")
splits_dir.mkdir(parents=True, exist_ok=True)
linked_splits = 0
for fname in ["train_val_folds.json", "test_public_uids.json", "test_private_uids.json"]:
    matches = glob.glob(f"/kaggle/input/**/{fname}", recursive=True)
    if matches:
        safe_symlink(Path(matches[0]), splits_dir / fname)
        linked_splits += 1
print(f"✅ Đã liên kết {linked_splits} tệp split vào {splits_dir}")

# 3. Liên kết sẵn pykt_kc_level/ nếu đã được upload kèm trong Kaggle Input
pykt_dir = proc_dir / "pykt_kc_level"
pykt_dir.mkdir(parents=True, exist_ok=True)
eedi_pykt_csvs = ["train_valid_sequences.csv", "test_unified.csv", "test_public.csv", "test_private.csv"]

cand_pykt = [
    p for p in glob.glob("/kaggle/input/**/test_unified.csv", recursive=True)
    if "pykt" in p.lower() or "eedi" in p.lower()
] or glob.glob("/kaggle/input/**/test_unified.csv", recursive=True)
if cand_pykt:
    src_pykt_dir = Path(cand_pykt[0]).parent
    print(f"🔍 Tìm thấy kho pykt_kc_level Eedi sẵn có tại: {src_pykt_dir}")
    for src_file in src_pykt_dir.glob("*.csv"):
        safe_symlink(src_file, pykt_dir / src_file.name)
    for src_file in src_pykt_dir.glob("*.json"):
        safe_symlink(src_file, pykt_dir / src_file.name)

# 4. Nếu thiếu pykt_kc_level CSVs, tự động tìm dữ liệu thô Eedi và chạy scripts/prepare_eedi_pykt_data.py
missing_pykt = [fn for fn in eedi_pykt_csvs if not (pykt_dir / fn).exists()]
if missing_pykt:
    print(f"⚙️ Chưa thấy sẵn {missing_pykt} trong /kaggle/input/ -> tự động tìm dữ liệu thô Eedi để chuyển đổi...")
    raw_base = Path("data/raw/Eedi NeurIPS 2020/data_extracted/data")
    raw_map = {
        "train_task_3_4.csv": raw_base / "train_data" / "train_task_3_4.csv",
        "test_public_task_4_more_splits.csv": raw_base / "test_data" / "test_public_task_4_more_splits.csv",
        "test_private_task_4_more_splits.csv": raw_base / "test_data" / "test_private_task_4_more_splits.csv",
        "answer_metadata_task_3_4.csv": raw_base / "metadata" / "answer_metadata_task_3_4.csv",
    }
    for fname, dst_p in raw_map.items():
        if not dst_p.exists():
            found = glob.glob(f"/kaggle/input/**/{fname}", recursive=True)
            if found:
                safe_symlink(Path(found[0]), dst_p)
    res_prep = subprocess.run([sys.executable, "scripts/prepare_eedi_pykt_data.py", "--dataset_config", "configs/dataset/eedi_pykt.yaml"])
    if res_prep.returncode != 0:
        raise RuntimeError("❌ Lỗi khi chạy scripts/prepare_eedi_pykt_data.py. Vui lòng kiểm tra dữ liệu Eedi gắn trong /kaggle/input/!")

# 5. Kiểm tra cuối cùng các tệp bắt buộc cho Eedi pyKT & EediTestContext
required_files = [
    pykt_dir / "train_valid_sequences.csv",
    pykt_dir / "test_unified.csv",
    pykt_dir / "test_public.csv",
    pykt_dir / "test_private.csv",
    proc_dir / "test_strata_masks.pt",
    proc_dir / "sequences_test.pt",
    proc_dir / "sequences_test_public.pt",
    proc_dir / "sequences_test_private.pt",
    proc_dir / "acd_v3_aux_metadata.pt",
]
missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError("❌ Thiếu các tệp Eedi bắt buộc:\n   " + "\n   ".join(missing))
print("🎉 Toàn bộ dữ liệu Eedi pyKT (5-Fold + Unified/Public/Private) và EediTestContext đã sẵn sàng!")

# 6. (Tùy chọn) Liên kết checkpoint/report sẵn có trong /kaggle/input/ nếu người dùng đính kèm
ckpt_eedi_root = Path("outputs/checkpoints/eedi")
ckpt_eedi_root.mkdir(parents=True, exist_ok=True)
restored_ckpts = 0
for model_prefix in ["dkt", "dkvmn", "sakt"]:
    for f_idx in range(5):
        folder = f"{model_prefix}_fold{f_idx}"
        for cand in glob.glob(f"/kaggle/input/**/{folder}/best_model.pt", recursive=True):
            if "eedi" in cand.lower() or "pykt" in cand.lower():
                src_folder = Path(cand).parent
                dst_folder = ckpt_eedi_root / folder
                dst_folder.mkdir(parents=True, exist_ok=True)
                for item in src_folder.glob("*"):
                    if item.is_file():
                        safe_symlink(item, dst_folder / item.name)
                        restored_ckpts += 1
                break
if restored_ckpts > 0:
    print(f"📦 Đã khôi phục {restored_ckpts} tệp checkpoint pyKT Eedi sẵn có vào {ckpt_eedi_root}")

In [ ]:
# [5] Chạy Sanity Check Unit & Integration Tests cho pyKT Baselines trên Eedi
import subprocess
import sys

cmd = [
    sys.executable, "-m", "pytest",
    "tests/test_eedi_pykt_baselines.py",
    "tests/test_pykt_protocol.py",
    "-v",
]
res = subprocess.run(cmd)
if res.returncode != 0:
    raise RuntimeError(f"❌ Sanity Check thất bại (exit code {res.returncode}). Vui lòng kiểm tra log phía trên.")
print("🎉 Toàn bộ bài kiểm thử Eedi pyKT (cô lập 5-Fold, chống rò rỉ 2-KC, khớp nối 1-đối-1 trên 3 phân vùng) đã PASSED 100%!")

## ⚙️ Cấu Hình Siêu Tham Số Huấn Luyện & Điều Phối 2x GPU T4 (Eedi `num_c = 57`)

Toàn bộ cấu hình tuân thủ `configs/dataset/eedi_pykt.yaml` và `configs/train_pykt_eedi.yaml`:
- **`NUM_EPOCHS = 50`**, **`PATIENCE = 10`** (dừng sớm theo Validation ROC-AUC), **`LEARNING_RATE = 1e-3`**, **`SEED = 42`**.
- **`BATCH_SIZE = 128`** (tối ưu cho `9.435` đoạn huấn luyện Eedi, tăng số bước cập nhật trọng số mỗi epoch mà vẫn ổn định trên T4).
- **`USE_DUAL_GPU`:** Khi có 2 GPU T4, Worker 0 (`cuda:0`) chạy Folds `[0, 1, 2]` song song với Worker 1 (`cuda:1`) chạy Folds `[3, 4]`.
- **`ALL_EEDI_SPLITS = True`:** Sau khi huấn luyện xong, tự động đánh giá trên cả **3 phân vùng kiểm thử** (`unified`: `124.960`, `public`: `74.276`, `private`: `50.684`) và cô lập đầu ra tại `outputs/{checkpoints,reports,logs}/eedi/`.

In [ ]:
# ==============================================================================
# [6] Thiết Lập Siêu Tham Số Toàn Cục (Eedi pyKT Baselines)
# ==============================================================================
import os
import torch

# 1. Mô hình mục tiêu: 'all' (chạy cả 3 mô hình DKT, DKVMN, SAKT), hoặc chọn riêng 'dkt', 'dkvmn', 'sakt'
TARGET_MODEL = "all"

# 2. Siêu tham số huấn luyện theo chuẩn configs/train_pykt_eedi.yaml
NUM_EPOCHS = 50
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
PATIENCE = 10
SEED = 42

# 3. Phân vùng đánh giá Eedi
TEST_SPLIT = "unified"        # Phân vùng chính ('unified', 'public', hoặc 'private')
ALL_EEDI_SPLITS = True        # True -> đánh giá đủ cả 3 phân vùng (unified, public, private)

# 4. Cấu hình phần cứng Dual GPU T4
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
USE_DUAL_GPU = (gpu_count >= 2)
DUAL_GPU_FLAG = "--dual-gpu" if USE_DUAL_GPU else ""
EVAL_SPLITS_FLAG = "--all_eedi_splits" if ALL_EEDI_SPLITS else f"--test_split {TEST_SPLIT}"
DEVICE_STR = "cuda:0" if torch.cuda.is_available() else "cpu"

# 5. Các Folds cần thực thi
FOLDS_TO_RUN = "0 1 2 3 4"

# Tạo sẵn thư mục cô lập riêng cho Eedi
os.makedirs("outputs/checkpoints/eedi", exist_ok=True)
os.makedirs("outputs/reports/eedi", exist_ok=True)
os.makedirs("outputs/reports/eedi/figures", exist_ok=True)
os.makedirs("outputs/logs/eedi", exist_ok=True)

print("🔧 CẤU HÌNH EEDI PYKT BASELINES ĐÃ THIẾT LẬP:")
print(f"   - Dataset: Eedi Task 3 & 4 (num_c = 57) | Splits: {'ALL (unified, public, private)' if ALL_EEDI_SPLITS else TEST_SPLIT}")
print(f"   - Mô hình mục tiêu: {TARGET_MODEL.upper()}")
print(f"   - Số Epochs: {NUM_EPOCHS} | Batch Size: {BATCH_SIZE} | LR: {LEARNING_RATE} | Patience: {PATIENCE} | Seed: {SEED}")
print(f"   - Số GPU phát hiện: {gpu_count} | Dual-GPU: {USE_DUAL_GPU} ('{DUAL_GPU_FLAG}')")
print(f"   - Folds sẽ chạy: {FOLDS_TO_RUN}")

## 🏋️ Huấn Luyện Tự Động 5-Fold & Đánh Giá Đa Phân Vùng (Master Runner)

Ô lệnh dưới đây gọi `scripts/run_5fold_pykt_baselines.py --dataset eedi` để:
1. **Pha 1 (Huấn luyện song song 2x GPU T4):** Worker 0 (`cuda:0`) chạy Folds `[0, 1, 2]` và Worker 1 (`cuda:1`) chạy Folds `[3, 4]` cho từng mô hình (`dkt`, `dkvmn`, `sakt`) với `num_c = 57`.
2. **Pha 2 (Đánh giá chống rò rỉ 2-KC & Late Mean Fusion):** Xuất mảng dự đoán `test_preds.npy`, `test_preds_public.npy`, `test_preds_private.npy` và báo cáo JSON/Markdown tại `outputs/reports/eedi/`.

In [ ]:
# [7] Chạy toàn bộ quy trình 5-Fold Cross-Validation cho các mô hình cơ sở pyKT trên Eedi
print(f"🚀 BẮT ĐẦU HUẤN LUYỆN 5-FOLD TRÊN EEDI CHO: {TARGET_MODEL.upper()}")
print(f"   Tham số: Epochs={NUM_EPOCHS}, BatchSize={BATCH_SIZE}, LR={LEARNING_RATE}, DualGPU={USE_DUAL_GPU}")

!python -u scripts/run_5fold_pykt_baselines.py \
    --dataset eedi \
    --dataset_config configs/dataset/eedi_pykt.yaml \
    --train_config configs/train_pykt_eedi.yaml \
    --model {TARGET_MODEL} \
    --epochs {NUM_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LEARNING_RATE} \
    --patience {PATIENCE} \
    --seed {SEED} \
    --folds {FOLDS_TO_RUN} \
    {EVAL_SPLITS_FLAG} \
    {DUAL_GPU_FLAG}

print("✅ Hoàn tất toàn bộ quy trình huấn luyện & đánh giá tổng hợp 5-Fold trên Eedi!")

## 🎯 Huấn Luyện Độc Lập Từng Mô Hình Cơ Sở (Tùy Chọn)

Nếu muốn chạy tách từng mô hình (`DKT`, `DKVMN`, `SAKT`) hoặc tinh chỉnh riêng Batch Size / Epochs trên Eedi, hãy dùng 3 cell dưới đây.

In [ ]:
# [8A] Huấn luyện riêng mô hình DKT trên Eedi (5 Folds, Dual-GPU T4)
DKT_EPOCHS = NUM_EPOCHS
DKT_BATCH_SIZE = BATCH_SIZE
DKT_FOLDS = FOLDS_TO_RUN

print(f"🚀 Bắt đầu huấn luyện DKT trên Eedi (Epochs: {DKT_EPOCHS}, Batch Size: {DKT_BATCH_SIZE}, Folds: {DKT_FOLDS})...")
!python -u scripts/run_5fold_pykt_baselines.py \
    --dataset eedi \
    --dataset_config configs/dataset/eedi_pykt.yaml \
    --train_config configs/train_pykt_eedi.yaml \
    --model dkt \
    --epochs {DKT_EPOCHS} \
    --batch_size {DKT_BATCH_SIZE} \
    --lr {LEARNING_RATE} \
    --patience {PATIENCE} \
    --seed {SEED} \
    --folds {DKT_FOLDS} \
    {EVAL_SPLITS_FLAG} \
    {DUAL_GPU_FLAG}
print("✅ Hoàn tất huấn luyện DKT trên Eedi!")

In [ ]:
# [8B] Huấn luyện riêng mô hình DKVMN trên Eedi (5 Folds, Dual-GPU T4)
DKVMN_EPOCHS = NUM_EPOCHS
DKVMN_BATCH_SIZE = BATCH_SIZE
DKVMN_FOLDS = FOLDS_TO_RUN

print(f"🚀 Bắt đầu huấn luyện DKVMN trên Eedi (Epochs: {DKVMN_EPOCHS}, Batch Size: {DKVMN_BATCH_SIZE}, Folds: {DKVMN_FOLDS})...")
!python -u scripts/run_5fold_pykt_baselines.py \
    --dataset eedi \
    --dataset_config configs/dataset/eedi_pykt.yaml \
    --train_config configs/train_pykt_eedi.yaml \
    --model dkvmn \
    --epochs {DKVMN_EPOCHS} \
    --batch_size {DKVMN_BATCH_SIZE} \
    --lr {LEARNING_RATE} \
    --patience {PATIENCE} \
    --seed {SEED} \
    --folds {DKVMN_FOLDS} \
    {EVAL_SPLITS_FLAG} \
    {DUAL_GPU_FLAG}
print("✅ Hoàn tất huấn luyện DKVMN trên Eedi!")

In [ ]:
# [8C] Huấn luyện riêng mô hình SAKT trên Eedi (5 Folds, Dual-GPU T4)
SAKT_EPOCHS = NUM_EPOCHS
SAKT_BATCH_SIZE = BATCH_SIZE
SAKT_FOLDS = FOLDS_TO_RUN

print(f"🚀 Bắt đầu huấn luyện SAKT trên Eedi (Epochs: {SAKT_EPOCHS}, Batch Size: {SAKT_BATCH_SIZE}, Folds: {SAKT_FOLDS})...")
!python -u scripts/run_5fold_pykt_baselines.py \
    --dataset eedi \
    --dataset_config configs/dataset/eedi_pykt.yaml \
    --train_config configs/train_pykt_eedi.yaml \
    --model sakt \
    --epochs {SAKT_EPOCHS} \
    --batch_size {SAKT_BATCH_SIZE} \
    --lr {LEARNING_RATE} \
    --patience {PATIENCE} \
    --seed {SEED} \
    --folds {SAKT_FOLDS} \
    {EVAL_SPLITS_FLAG} \
    {DUAL_GPU_FLAG}
print("✅ Hoàn tất huấn luyện SAKT trên Eedi!")

## 📊 Đánh Giá Đa Phân Vùng Eedi (`unified`, `public`, `private`) & Đối Sánh Với `ACD-MMKTv3`

Cell dưới đây chạy đánh giá (hoặc nạp nhanh từ `test_preds*.npy` qua `--reuse_preds`) trên cả 3 phân vùng kiểm thử Eedi và hiển thị động các bảng hiệu năng thực tế (Toàn cục, Hiệu chuẩn, Phân tầng Sư phạm & Đối sánh với `ACD-MMKTv3` nếu báo cáo/checkpoint `ACD-MMKTv3` có mặt trên đĩa).

In [ ]:
# [9] Đánh giá trên cả 3 phân vùng Eedi (unified, public, private) & hiển thị bảng tổng hợp thực tế
import json
import os
import subprocess
import sys
import pandas as pd
from IPython.display import display, Markdown

splits_to_eval = ["unified", "public", "private"] if ALL_EEDI_SPLITS else [TEST_SPLIT]

for sp in splits_to_eval:
    print(f"\n🔍 Đang đánh giá phân vùng '{sp}' trên Eedi...")
    cmd = [
        sys.executable, "-u", "scripts/evaluate_pykt_baselines.py",
        "--dataset", "eedi",
        "--dataset_config", "configs/dataset/eedi_pykt.yaml",
        "--train_config", "configs/train_pykt_eedi.yaml",
        "--model", "all",
        "--test_split", sp,
        "--device", DEVICE_STR,
        "--reuse_preds",
    ]
    res = subprocess.run(cmd)
    if res.returncode != 0:
        print(f"⚠️ Cảnh báo: đánh giá phân vùng '{sp}' trả về mã lỗi {res.returncode}.")

# Tùy chọn: Nếu đã có sẵn checkpoints ACD-MMKTv3 trên Eedi, tái tổng hợp báo cáo đối sánh (chế độ --eval-only)
full_v3_ready = all(
    os.path.exists(f"outputs/checkpoints/eedi/acd_mmkt_v3_full_5fold/acd_mmkt_v3_fold{f}/best_model.pt")
    for f in range(5)
)
if full_v3_ready:
    print("\n🔗 Phát hiện checkpoints Full ACD-MMKTv3 trên Eedi -> Tái tổng hợp báo cáo đối sánh (--eval-only)...")
    subprocess.run([sys.executable, "-u", "scripts/run_5fold_acd_mmkt_v3.py", "--dataset", "eedi", "--eval-only"])
    ab_ready = all(
        os.path.exists(f"outputs/checkpoints/eedi/acd_mmkt_v3_ablation1_wo_all_multimodal_5fold/acd_mmkt_v3_fold{f}/best_model.pt")
        for f in range(5)
    )
    if ab_ready:
        for sp in splits_to_eval:
            subprocess.run([
                sys.executable, "-u", "scripts/run_acd_v3_modality_ablation_study.py",
                "--dataset", "eedi", "--test-split", sp, "--eval-only", "--device", DEVICE_STR,
            ])

# Hiển thị bảng kết quả cho từng phân vùng
for sp in splits_to_eval:
    tag = "" if sp == "unified" else f"_{sp}"
    json_path = f"outputs/reports/eedi/pykt_baselines_comprehensive_evaluation{tag}.json"
    if not os.path.exists(json_path):
        continue
    with open(json_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)

    display(Markdown(f"### 🏆 Bảng 1 ({sp.upper()}): Hiệu Năng Toàn Cục 5-Fold Trên Eedi"))
    rows = []
    for m_name in ["dkt", "dkvmn", "sakt"]:
        if m_name not in eval_data:
            continue
        d = eval_data[m_name]
        em = d.get("ensemble_metrics", {})
        ext = em.get("overall_extended", {})
        rows.append({
            "Model": f"{m_name.upper()} (pyKT)",
            "Params": f"{d.get('num_parameters', 0):,}",
            "5-Fold Mean AUC": f"{d.get('mean_test_auc', 0):.4f} ± {d.get('std_test_auc', 0):.4f}",
            "5-Fold Mean ACC": f"{d.get('mean_test_acc', 0):.4f} ± {d.get('std_test_acc', 0):.4f}",
            "Ens. AUC": f"{em.get('overall', {}).get('auc', 0):.4f}",
            "Ens. ACC": f"{em.get('overall', {}).get('acc', 0):.4f}",
            "RMSE": f"{ext.get('rmse', 0):.4f}" if ext else "--",
            "ECE": f"{ext.get('ece', 0):.4f}" if ext else "--",
            "Macro-AUC": f"{ext.get('student_macro_auc', 0):.4f}" if ext else "--",
        })

    # Nạp động kết quả ACD-MMKTv3 nếu có sẵn trong outputs/reports/eedi/
    comp_path = f"outputs/reports/eedi/acd_mmkt_v3_modality_ablation_study_comprehensive{tag}.json"
    if os.path.exists(comp_path):
        with open(comp_path, "r", encoding="utf-8") as f:
            comp = json.load(f)
        fm = comp.get("full_model", {})
        f_ens = fm.get("ensemble_metrics", {})
        f_mean = fm.get("mean_metrics", {})
        rows.append({
            "Model": "ACD-MMKTv3 (Full)",
            "Params": "762,376",
            "5-Fold Mean AUC": f"{f_mean.get('auc', 0):.4f} ± {f_mean.get('auc_std', 0):.4f}",
            "5-Fold Mean ACC": "--",
            "Ens. AUC": f"{f_ens.get('auc', 0):.4f}",
            "Ens. ACC": f"{f_ens.get('acc', 0):.4f}",
            "RMSE": f"{f_ens.get('rmse', 0):.4f}",
            "ECE": f"{f_ens.get('ece', 0):.4f}",
            "Macro-AUC": f"{f_ens.get('student_macro_auc', 0):.4f}",
        })
        ab1 = comp.get("ablations", {}).get("1", {})
        if ab1:
            ab1_m = ab1.get("mean_metrics", {})
            ab1_e = ab1.get("ensemble_metrics", {})
            rows.append({
                "Model": "Ablation 1 (Pure KC)",
                "Params": f"{ab1.get('total_params', 0):,}",
                "5-Fold Mean AUC": f"{ab1_m.get('auc', 0):.4f} ± {ab1_m.get('auc_std', 0):.4f}",
                "5-Fold Mean ACC": f"{ab1_m.get('acc', 0):.4f} ± {ab1_m.get('acc_std', 0):.4f}",
                "Ens. AUC": f"{ab1_e.get('auc', 0):.4f}",
                "Ens. ACC": f"{ab1_e.get('acc', 0):.4f}",
                "RMSE": f"{ab1_e.get('rmse', 0):.4f}",
                "ECE": f"{ab1_e.get('ece', 0):.4f}",
                "Macro-AUC": f"{ab1_e.get('student_macro_auc', 0):.4f}",
            })
    display(pd.DataFrame(rows))

    display(Markdown(f"#### 🖼️ Bảng 2 ({sp.upper()}): Phân Tầng Sư Phạm Trên Tập Eedi"))
    strat_rows = []
    for m_name in ["dkt", "dkvmn", "sakt"]:
        if m_name not in eval_data:
            continue
        es = eval_data[m_name].get("ensemble_metrics", {}).get("eedi_strata", {})
        if not es:
            continue
        strat_rows.append({
            "Model": m_name.upper(),
            "Overall": f"{es.get('overall', {}).get('auc', 0):.4f}",
            "Diagram (m_q=1)": f"{es.get('q_stem_image', {}).get('auc', 0):.4f}",
            "Text-Only (m_q=0)": f"{es.get('q_text_only', {}).get('auc', 0):.4f}",
            "Remedial (r_{t-1}=0)": f"{es.get('after_incorrect', {}).get('auc', 0):.4f}",
            "Cold-Start (<=10)": f"{es.get('freq_rare_le10', {}).get('auc', 0):.4f}",
            "Frequent (>100)": f"{es.get('freq_frequent_gt100', {}).get('auc', 0):.4f}",
            "Multi-KC": f"{es.get('multi_kc', {}).get('auc', 0):.4f}",
        })
    if strat_rows:
        display(pd.DataFrame(strat_rows))

## 📈 Trực Quan Hóa Kết Quả Thực Tế Trên Eedi (300 DPI)

In [ ]:
# [10] Vẽ biểu đồ phân tích từ kết quả thực tế của Eedi (100% động, không dùng số giả)
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs("outputs/reports/eedi/figures", exist_ok=True)
eval_json = "outputs/reports/eedi/pykt_baselines_comprehensive_evaluation.json"
if not os.path.exists(eval_json):
    print("ℹ️ Chưa có outputs/reports/eedi/pykt_baselines_comprehensive_evaluation.json — hãy chạy Cell [7] hoặc Cell [9] trước.")
else:
    with open(eval_json, "r", encoding="utf-8") as f:
        data = json.load(f)

    sns.set_theme(style="whitegrid", font="sans-serif")
    plt.rcParams.update({"font.size": 11, "axes.labelsize": 11.5, "axes.titlesize": 12.5})

    labels, mean_aucs, std_aucs, ens_aucs, ece_vals = [], [], [], [], []
    diag_aucs, txt_aucs, rem_aucs, rare_aucs, freq_aucs, multi_aucs = [], [], [], [], [], []
    for m_key in ["dkt", "dkvmn", "sakt"]:
        if m_key not in data:
            continue
        d = data[m_key]
        em = d["ensemble_metrics"]
        es = em.get("eedi_strata", {})
        ext = em.get("overall_extended", {})
        labels.append(f"{m_key.upper()} (pyKT)")
        mean_aucs.append(d["mean_test_auc"])
        std_aucs.append(d["std_test_auc"])
        ens_aucs.append(em["overall"]["auc"])
        ece_vals.append(ext.get("ece", 0.0))
        diag_aucs.append(es.get("q_stem_image", {}).get("auc", 0.0))
        txt_aucs.append(es.get("q_text_only", {}).get("auc", 0.0))
        rem_aucs.append(es.get("after_incorrect", {}).get("auc", 0.0))
        rare_aucs.append(es.get("freq_rare_le10", {}).get("auc", 0.0))
        freq_aucs.append(es.get("freq_frequent_gt100", {}).get("auc", 0.0))
        multi_aucs.append(es.get("multi_kc", {}).get("auc", 0.0))

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 11))
    x = np.arange(len(labels))
    w = 0.35

    # (a) 5-Fold Mean vs Ensemble AUC
    b1 = ax1.bar(x - w/2, mean_aucs, yerr=std_aucs, capsize=4, width=w, label="5-Fold Mean ± Std", color="#aec7e8", edgecolor="black", linewidth=0.7)
    b2 = ax1.bar(x + w/2, ens_aucs, width=w, label="5-Fold Soft Ensemble", color="#1f77b4", edgecolor="black", linewidth=0.7)
    for bar in list(b1) + list(b2):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001, f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=9)
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels)
    ax1.set_title("(a) Eedi Unified: 5-Fold Mean vs Soft Ensemble ROC-AUC", fontweight="bold")
    ax1.set_ylabel("ROC-AUC")
    ax1.legend()

    # (b) Split Comparison (Unified / Public / Private)
    split_series = {}
    for sp, tag in [("Unified", ""), ("Public", "_public"), ("Private", "_private")]:
        sp_path = f"outputs/reports/eedi/pykt_baselines_comprehensive_evaluation{tag}.json"
        if os.path.exists(sp_path):
            with open(sp_path, "r", encoding="utf-8") as sf:
                sd = json.load(sf)
            split_series[sp] = [sd.get(k, {}).get("ensemble_metrics", {}).get("overall", {}).get("auc", np.nan) for k in ["dkt", "dkvmn", "sakt"] if k in data]
    width_s = 0.25
    n_splits = max(1, len(split_series))
    for idx_s, (sp_name, vals) in enumerate(split_series.items()):
        offset = (idx_s - (n_splits - 1) / 2.0) * width_s
        ax2.bar(x + offset, vals, width=width_s, label=sp_name, edgecolor="black", linewidth=0.7)
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels)
    ax2.set_title("(b) Ensemble ROC-AUC Across Eedi Test Splits", fontweight="bold")
    ax2.set_ylabel("ROC-AUC")
    ax2.legend()

    # (c) Visual vs Text vs Remedial
    ax3.plot(labels, diag_aucs, marker="o", linewidth=2, label="Stem Image (m_q=1)")
    ax3.plot(labels, txt_aucs, marker="s", linewidth=2, linestyle="--", label="Text-Only (m_q=0)")
    ax3.plot(labels, rem_aucs, marker="^", linewidth=2, label="Remedial (r_{t-1}=0)")
    ax3.set_title("(c) Modality & Remedial Learning Slices (Eedi Unified)", fontweight="bold")
    ax3.set_ylabel("ROC-AUC")
    ax3.legend()

    # (d) Cold-Start vs Frequent vs Multi-KC
    ax4.plot(labels, rare_aucs, marker="o", linewidth=2, label="Cold-Start (N_q <= 10)")
    ax4.plot(labels, freq_aucs, marker="s", linewidth=2, label="Frequent (N_q > 100)")
    ax4.plot(labels, multi_aucs, marker="d", linewidth=2, label="Multi-KC (2 skills)")
    ax4.set_title("(d) Frequency & Skill Complexity Slices (Eedi Unified)", fontweight="bold")
    ax4.set_ylabel("ROC-AUC")
    ax4.legend()

    plt.tight_layout()
    fig_path = "outputs/reports/eedi/figures/pykt_baselines_eedi_comparative_analysis.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"✅ Biểu đồ phân tích Eedi đã được lưu tại: {fig_path}")

## 📦 Đóng Gói Kết Quả Eedi Thành Tệp ZIP Để Tải Về

Nén toàn bộ checkpoints (`outputs/checkpoints/eedi`), báo cáo (`outputs/reports/eedi`) và nhật ký (`outputs/logs/eedi`) thành tệp ZIP duy nhất tại `/kaggle/working/pykt_baselines_eedi_results.zip`.

In [ ]:
# [11] Đóng gói Checkpoints, Reports và Logs Eedi (không chạm tới kết quả XES3G5M)
import os

zip_path = "/kaggle/working/pykt_baselines_eedi_results.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

!zip -q -r {zip_path} outputs/checkpoints/eedi outputs/reports/eedi outputs/logs/eedi
zip_size_mb = os.path.getsize(zip_path) / (1024**2)
print(f"🎉 Đã nén thành công kết quả Eedi vào: {zip_path} ({zip_size_mb:.2f} MB)")
print("👉 Bạn có thể tải trực tiếp từ tab 'Output' bên phải giao diện Kaggle!")